In [1]:
import sys
PROJECT_ROOT = r"C:\\Dev\\Synthetic-Data-Generation-For-Cardiovascular-Disease-Risk-Prediction\\Final_models"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
import torch
import numpy as np
from tslearn.metrics import dtw
from BiLSTM_CNN import Generator as bilstm_gen
from ConvTranspose1D import Generator as convtran_gen
from UpsampleAndConv1D import Generator as upconv_gen
from preprocessing_utils import per_lead_minmax_scaling
from torch.utils.data import DataLoader, TensorDataset
from collections.abc import Callable
from ignite.metrics import MaximumMeanDiscrepancy as MMD
import os

%matplotlib inline


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

c:\Dev\Synthetic-Data-Generation-For-Cardiovascular-Disease-Risk-Prediction\.venv\Lib\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Loading the data into a pytorch dataloader for checking GAN metrics

In [2]:
if os.path.exists("../../fine_tune_data.npy"):
    data = np.load("../../fine_tune_data.npy", allow_pickle=True)
    segments = [item[0] for item in data]
    ecg_dataset = np.stack(segments)
    normalized_data, lead_mins, lead_maxs = per_lead_minmax_scaling(ecg_dataset)
    labels = [item[1] for item in data]
labels = np.array(labels)
normalized_data = np.array(normalized_data)
dataset_tensor = torch.tensor(normalized_data, dtype=torch.float32)
labels_tensor = torch.tensor(labels, dtype=torch.long).unsqueeze(1)
dataloader = DataLoader(TensorDataset(dataset_tensor,labels_tensor),batch_size=128, shuffle=True,drop_last=True)

### MVDTW Metric Evaluation functions

In [3]:
def mvdtw_pair(real_seq:np.ndarray, fake_seq:np.ndarray) -> float:
    return dtw(real_seq,fake_seq)

def evaluate_generator_mvdtw(generator: bilstm_gen | convtran_gen | upconv_gen, dataloader: DataLoader, latent_dim: int, device: torch.device,mvdtw_fn:Callable[[np.ndarray,np.ndarray], float], max_batches: int | None = None):
    generator.eval() 
    total_dist = 0.0
    n_pairs = 0
    with torch.no_grad():
        for b_idx, (real_ecg, labels) in enumerate(dataloader):
            if max_batches is not None and b_idx >= max_batches:
                break
            real_ecg: torch.Tensor = real_ecg.to(device)
            labels: torch.Tensor = labels.to(device)
            B = real_ecg.size(0)
            noise = torch.randn(B, latent_dim, device=device)
            fake_ecg:torch.Tensor = generator(noise,labels)

            if real_ecg.shape[1] == 640 and real_ecg.shape[2] == 3:
                real_seq = real_ecg
            elif real_ecg.shape[1] == 3 and real_ecg.shape[2] == 640:
                real_seq = real_ecg.permute(0,2,1)
            else:
                raise ValueError(f"Unexpected real_ecg shape: {real_ecg.shape}")

            if fake_ecg.shape[1] == 640 and fake_ecg.shape[2] == 3:
                fake_seq = fake_ecg
            elif fake_ecg.shape[1] == 3 and fake_ecg.shape[2] == 640:
                fake_seq = fake_ecg.permute(0,2,1)
            else:
                raise ValueError(f"Unexpected fake_ecg shape: {fake_ecg.shape}")
            
            real_np = real_seq.cpu().numpy()
            fake_np = fake_seq.cpu().numpy()

            for i in range(B):
                d = mvdtw_fn(real_np[i],fake_np[i])
                total_dist += float(d)
                n_pairs += 1
    avg_mvdtw = total_dist / max(n_pairs,1)
    return avg_mvdtw

### MMD Metric Evaluation functions

In [4]:
def flatten_ecg_for_mmd(x: torch.Tensor) -> torch.Tensor:
    if x.dim() != 3:
        raise ValueError(f"Expected 3D tensor, got {x.shape}")
    
    if x.shape[1] == 3 and x.shape[2]==640:
        x_ch = x
    elif x.shape[1]==640 and x.shape[2]==3:
        x_ch = x.permute(0,2,1)
    else:
        raise ValueError(f"Unexpected ECG shape for flattening: {x.shape}")
    
    return x_ch.reshape(x_ch.size(0), -1)

def evaluate_generator_mmd(generator : bilstm_gen | convtran_gen | upconv_gen, dataloader:DataLoader,latent_dim:int,device:torch.device, kernel_var=1.0,max_batches=None):
    generator.eval()
    mmd = MMD(var=kernel_var,device=device)
    mmd.reset()
    with torch.no_grad():
        for b_idx, (real_ecg, labels) in enumerate(dataloader):
            if max_batches is not None and b_idx >= max_batches:
                break
            real_ecg:torch.Tensor = real_ecg.to(device)
            labels:torch.Tensor = labels.to(device)
            B=real_ecg.size(0)
            noise = torch.randn(B,latent_dim,device=device)
            fake_ecg:torch.Tensor = generator(noise,labels)
            real_flat = flatten_ecg_for_mmd(real_ecg).to(device=device, dtype=torch.float32)
            fake_flat = flatten_ecg_for_mmd(fake_ecg).to(device=device, dtype=torch.float32)
            mmd.update((fake_flat, real_flat))
    mmd_value = mmd.compute()
    if isinstance(mmd_value, torch.Tensor):
        mmd_value = mmd_value.item()
    return mmd_value

## Metrics for Models Trained Without MVDTW Loss Term

### BiLSTM-CNN Model Metrics

In [5]:
model = torch.load(
    "models/BiLSTM_CNN_CWGAN/Model_0_GP_10.0_DTW_0.0/Model.pth", weights_only=False)
generator_bilstm = bilstm_gen().to(device)
generator_bilstm.load_state_dict(model['gen_state_dict'])

mvdtw_bilstm = evaluate_generator_mvdtw(generator_bilstm,dataloader=dataloader,latent_dim=100,mvdtw_fn=mvdtw_pair,device=device)
print(f"MVDTW: {mvdtw_bilstm:.4f}")

c:\Dev\Synthetic-Data-Generation-For-Cardiovascular-Disease-Risk-Prediction\.venv\Lib\site-packages\torch\nn\modules\conv.py:366: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\Convolution.cpp:1032.)
  return F.conv1d(


MVDTW: 2.0768


In [6]:
mmd_bilstm = evaluate_generator_mmd(generator=generator_bilstm,dataloader=dataloader,latent_dim=100,device=device)
print(f"MMD: {mmd_bilstm:.4f}")

MMD: 0.0596


### Transposed-Convolution Model Metrics

In [7]:
model = torch.load(
    "models/DCNN_WGAN/Model_0_GP_10.0_DTW_0.0/Model.pth", weights_only=False)
generator_tran = convtran_gen().to(device)
generator_tran.load_state_dict(model['gen_state_dict'])

mvdtw_dcnn = evaluate_generator_mvdtw(generator_tran,dataloader=dataloader,latent_dim=100,mvdtw_fn=mvdtw_pair,device=device)
print(f"MVDTW: {mvdtw_dcnn:.4f}")

MVDTW: 2.0714


In [8]:
mmd_bilstm = evaluate_generator_mmd(generator=generator_tran,dataloader=dataloader,latent_dim=100,device=device)
print(f"MMD: {mmd_bilstm:.4f}")

MMD: 0.0627


### Upsample-Convolution Model Metrics

In [9]:
model = torch.load(
    "models/UpsampleAndCNN_CWGAN/Model_0_GP_10.0_DTW_0.0/Model.pth", weights_only=False)
generator_upconv = upconv_gen().to(device)
generator_upconv.load_state_dict(model['gen_state_dict'])

mvdtw_bilstm = evaluate_generator_mvdtw(generator_upconv,dataloader=dataloader,latent_dim=100,mvdtw_fn=mvdtw_pair,device=device)
print(f"MVDTW: {mvdtw_bilstm:.4f}")

MVDTW: 2.0616


In [10]:
mmd_bilstm = evaluate_generator_mmd(generator=generator_upconv,dataloader=dataloader,latent_dim=100,device=device)
print(f"MMD: {mmd_bilstm:.4f}")

MMD: 0.0875


## Metrics for Models Trained With MVDTW Loss Term

### BiLSTM-CNN Model Metrics

In [ ]:
model = torch.load(
    "models/BiLSTM_CNN_CWGAN/Model_1_GP_10.0_DTW_1.0/Model.pth", weights_only=False)
generator_bilstm_mvdtw = bilstm_gen().to(device)
generator_bilstm_mvdtw.load_state_dict(model['gen_state_dict'])

mvdtw_bilstm_dtw = evaluate_generator_mvdtw(generator_bilstm_mvdtw,dataloader=dataloader,latent_dim=100,mvdtw_fn=mvdtw_pair,device=device)
print(f"MVDTW: {mvdtw_bilstm_dtw:.4f}")

MVDTW: 1.9830


In [12]:
mmd_bilstm_dtw = evaluate_generator_mmd(generator=generator_bilstm_mvdtw,dataloader=dataloader,latent_dim=100,device=device)
print(f"MMD: {mmd_bilstm_dtw:.4f}")

MMD: 0.0534


### Transposed-Convolution Model Metrics

In [13]:
model = torch.load(
    "models/DCNN_WGAN/Model_1_GP_10.0_DTW_1.0/Model.pth", weights_only=False)
generator_tran_mvdtw = convtran_gen().to(device)
generator_tran_mvdtw.load_state_dict(model['gen_state_dict'])

mvdtw_tran_dtw = evaluate_generator_mvdtw(generator_tran_mvdtw,dataloader=dataloader,latent_dim=100,mvdtw_fn=mvdtw_pair,device=device)
print(f"MVDTW: {mvdtw_tran_dtw:.4f}")

MVDTW: 2.0410


In [14]:
mmd_tran_dtw = evaluate_generator_mmd(generator=generator_tran_mvdtw,dataloader=dataloader,latent_dim=100,device=device)
print(f"MMD: {mmd_tran_dtw:.4f}")

MMD: 0.0822


### Upsample-Convolution Model Metrics

In [15]:
model = torch.load(
    "models/UpsampleAndCNN_CWGAN/Model_1_GP_10.0_DTW_1.0/Model.pth", weights_only=False)
generator_upconv_mvdtw = upconv_gen().to(device)
generator_upconv_mvdtw.load_state_dict(model['gen_state_dict'])

mvdtw_bilstm = evaluate_generator_mvdtw(generator_upconv_mvdtw,dataloader=dataloader,latent_dim=100,mvdtw_fn=mvdtw_pair,device=device)
print(f"MVDTW: {mvdtw_bilstm:.4f}")

MVDTW: 2.0203


In [16]:
mmd_bilstm = evaluate_generator_mmd(generator=generator_upconv_mvdtw,dataloader=dataloader,latent_dim=100,device=device)
print(f"MMD: {mmd_bilstm:.4f}")

MMD: 0.0450
